# Модуль 07 · Урок 3. ETL-пайплайны и устойчивость к повторному запуску

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Extract → Transform → Load
ETL означает **извлечение** данных из источника, **преобразование** по проверяемым правилам и **загрузку** результата. В Python правильнее разделить три этапа на отдельные функции. Тогда тестирование трансформаций не требует базы данных или API: на вход передаётся DataFrame, на выходе получается новый DataFrame. Настоящие источники могут быть медленными и ненадёжными, поэтому их особенно полезно отделять.

In [ ]:
import pandas as pd
raw = pd.DataFrame({'qty':['2','3'], 'price':['10.0','bad']})
def transform(raw_df):
    return raw_df.assign(qty_num=pd.to_numeric(raw_df.qty,errors='coerce'),price_num=pd.to_numeric(raw_df.price,errors='coerce'))
print(transform(raw))

## 2. Прозрачность последовательности
Pipeline должен явно описывать шаги: нормализация названий → преобразование типов → проверка данных → агрегация → экспорт. Для преобразований используйте `assign`, `pipe`, именованные функции. `df.pipe(clean_text).pipe(parse_numbers)` удобен тем, что каждый шаг легко проверить отдельно и заменить. Избегайте скрытых глобальных переменных и зависимостей от текущего каталога запуска.

## 3. Идемпотентность загрузки
Повторный запуск не должен удваивать продажи. На уровне pandas можно сохранить последнюю версию записи по ключу или сравнить новые ID с уже обработанными. В базе данных профессиональное решение обычно использует UNIQUE и UPSERT/ON CONFLICT. Если источник присылает одинаковые события с разным временем обновления, определите, какое событие считается актуальным.

## 4. Логирование, ошибки, структура проекта
Практичная структура: `src/extract.py`, `src/transform.py`, `src/load.py`, `tests/`, `config/`. Отдельно фиксируйте время выполнения, число входных и выходных строк, процент ошибок. При исключениях не надо бездумно подавлять ошибки `except Exception: pass`: потерянный сигнал способен исказить отчёт. Данные с ошибками полезно сохранять в карантине.

**Проверьте себя:** зачем отделять transform от extract? Что значит идемпотентность? Почему нельзя просто перезапустить load при любом исключении без проверки результата?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m7_11 · etl transform

Подготовьте заказы для загрузки: создайте `revenue`=price*qty*(1-discount/100), выберите `order_id`,`client_id`,`revenue`,`status`; не сохраняйте отменённые, сбросьте индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_11: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_11 = expected_for("m7_11", df=df)


In [ ]:
# m7_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 4 столбца; колонки: `order_id`, `client_id`, `revenue`, `status`.

```text
  order_id  client_id  revenue status
0  demo_o1          1    254.0   paid
1  demo_o2          2    222.3    new
2  demo_o3          1    368.6   paid
3  demo_o4          3    317.0   paid
4  demo_o6          4      0.0    new
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_11", result, _expected_m7_11)

### m7_12 · etl dedup

Удалите повторы по `order_id`, оставив последнее появление, затем отсортируйте по order_id и сбросьте индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_12: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6', 'demo_o1', 'demo_o2'],
    'client_id': [1, 2, 1, 3, 2, 4, 1, 2],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань', 'Москва', 'Тверь'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай', 'чай', 'кофе'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0, 127.0, 247.0],
    'qty': [2, 1, 4, 1, 3, 0, 2, 1],
    'discount': [0, 10, 5, 0, 20, 0, 0, 10],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new', 'paid', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05', '2026-01-03', '2026-01-04'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None, '  Быстрая доставка  ', 'без звонка'],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_12 = expected_for("m7_12", df=df)


In [ ]:
# m7_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
0  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
1  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
2  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
3  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
4  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
5  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_12", result, _expected_m7_12)

### m7_13 · etl parse events

Подготовьте логи событий: добавьте timestamp из текстового ts через to_datetime, а value приведите к float. Остальные поля сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `user`, `event`, `ts`, `value`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_13: используются переменные df

df = pd.DataFrame({
    'user': ['a', 'a', 'b', 'a', 'b', 'c'],
    'event': ['visit', 'buy', 'visit', 'visit', 'buy', 'visit'],
    'ts': ['2026-01-01 09:00', '2026-01-02 10:30', '2026-01-01 12:00', '2026-01-04 09:00', '2026-01-05 11:00', '2026-01-06 19:30'],
    'value': [0, 150, 0, 0, 75, 0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_13 = expected_for("m7_13", df=df)


In [ ]:
# m7_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 5 столбцов; колонки: `user`, `event`, `ts`, `value`, `timestamp`.

```text
  user  event               ts  value        timestamp
0    a  visit  2026-01-01 0...    0.0 2026-01-01 09...
1    a    buy  2026-01-02 1...  150.0 2026-01-02 10...
2    b  visit  2026-01-01 1...    0.0 2026-01-01 12...
3    a  visit  2026-01-04 0...    0.0 2026-01-04 09...
4    b    buy  2026-01-05 1...   75.0 2026-01-05 11...
5    c  visit  2026-01-06 1...    0.0 2026-01-06 19...
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_13", result, _expected_m7_13)

### m7_14 · etl quarantine

Разделите заказы на valid и invalid: valid при qty >= 0, price >= 0 и статусе из {paid,new,cancelled}. Получите кортеж DataFrame без сброса индексов.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_14: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, -1],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_14 = expected_for("m7_14", df=df)


In [ ]:
# m7_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Кортеж/список из 2 объектов: DataFrame (5, 10), DataFrame (1, 10).

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_14", result, _expected_m7_14)

### m7_15 · etl report

Получите dict `input_rows`, `output_rows`, `rejected_rows`: разница по числу строк входа и выхода (не отрицательная).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `raw` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `clean` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `raw`, `clean`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_15: используются переменные raw, clean

raw = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

clean = pd.DataFrame({
    'order_id': ['demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 3, 2, 4],
    'city': ['Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'книги', 'кофе', 'чай'],
    'price': [97.0, 317.0, 57.0, 147.0],
    'qty': [4, 1, 3, 0],
    'discount': [5, 0, 20, 0],
    'status': ['paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['Срочно', 'подарок', '', None],
}, index=['a', 'd', 'x', 'q'])

print("raw — демонстрационные данные:")

display(raw)

print("clean — демонстрационные данные:")

display(clean)


# Снимок правильного результата для автопроверки.
_expected_m7_15 = expected_for("m7_15", raw=raw, clean=clean)


In [ ]:
# m7_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: raw, clean
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'input_rows': 6, 'output_rows': 4, 'rejected_rows': 2}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_15", result, _expected_m7_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()